In [ ]:
from pathlib import Path                                    
import torch                                                
from sentence_transformers import SentenceTransformer      

if torch.cuda.is_available():                                
    device = "cuda"                                         
elif torch.backends.mps.is_available():                    
    device = "mps"                                           
else:                                                        
    device = "cpu"                                           
print("device:", device)                                     

ROOT = Path.cwd()                                            
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:  
    ROOT = ROOT.parent                                     

model = SentenceTransformer("BAAI/bge-m3", cache_folder=str(ROOT / "models"), device=device)  

docs = {                                                     
    "breakfast.txt": "Breakfast is served from 7:00 AM to 10:30 AM in the main restaurant.",
    "spa.txt": "The spa is open daily. A 60 minute treatment is 3,500 EGP.",
    "pets.txt": "Pets are not allowed in the hotel.",
}
names = list(docs.keys())                                  
texts = list(docs.values())                                 

doc_vecs = model.encode(texts, normalize_embeddings=True)    
print(doc_vecs.shape)                                        

questions = [                                                
    "الفطار لحد إمتى؟",                                       
    "Can I bring my dog?",                                  
    "عايز أعمل مساج",                                         
    "What's the weather tomorrow?",                         
]

for q in questions:                                         
    q_vec = model.encode([q], normalize_embeddings=True)   
    scores = model.similarity(q_vec, doc_vecs)[0]            
    print(q)                                                
    for i in range(len(names)):                              
        print("   ", round(float(scores[i]), 2), names[i])  

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 11:13:59.481000 38895 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


device: mps


Could not reach the Hub ([Errno 8] nodename nor servname provided, or not known). Using cached commit hash for 'BAAI/bge-m3'.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 49727.79it/s]


(3, 1024)
الفطار لحد إمتى؟
    0.61 breakfast.txt
    0.42 spa.txt
    0.42 pets.txt
Can I bring my dog?
    0.43 breakfast.txt
    0.41 spa.txt
    0.66 pets.txt
عايز أعمل مساج
    0.44 breakfast.txt
    0.55 spa.txt
    0.43 pets.txt
What's the weather tomorrow?
    0.43 breakfast.txt
    0.38 spa.txt
    0.43 pets.txt


In [ ]:
import time                                               

model.half()                                                
model.max_seq_length = 512                                   

texts = []                                                   
for path in sorted((ROOT / "knowledge_base").glob("*.txt")): 
    texts.append(path.read_text(encoding="utf-8"))           

t = time.time()                                              
model.encode(["test"], normalize_embeddings=True)           
print("1 sentence:", round(time.time() - t, 1), "s")        

t = time.time()                                             
model.encode(texts[:4], normalize_embeddings=True, batch_size=4)  
print("4 files:", round(time.time() - t, 1), "s")           

1 sentence: 1.7 s
4 files: 1.5 s


In [ ]:
import time                                                 

model.half()                                                
model.max_seq_length = 512                                  

KB_DIR = ROOT / "knowledge_base"                         

names = []                                                  
texts = []                                                   
for path in sorted(KB_DIR.glob("*.txt")):                   
    names.append(path.name)                                  
    texts.append(path.read_text(encoding="utf-8"))           
print(len(names), "files")                                   

doc_vecs = model.encode(texts, normalize_embeddings=True, batch_size=4, show_progress_bar=True) 
print(doc_vecs.shape)                                        

def dense_search(question, top_k=3):                        
    q_vec = model.encode([question], normalize_embeddings=True) 
    scores = model.similarity(q_vec, doc_vecs)[0]            
    results = []                                            
    for i in range(len(names)):                              
        results.append((round(float(scores[i]), 2), names[i])) 
    results.sort(reverse=True)                               
    return results[:top_k]                                   

questions = [                                           
    "What time is breakfast?",
    "How much is the spa?",
    "Can I bring my dog?",
    "الفطار لحد إمتى؟",
    "ممكن أدخل الأوضة بدري؟",                                
    "What's the weather tomorrow?",                          
]

for q in questions:                                         
    print(q)                                                
    for score, name in dense_search(q):                   
        print("   ", score, name)                            

start = time.time()                                          
dense_search("الفطار لحد إمتى؟")                            
print("time:", round(time.time() - start, 3), "seconds")     

44 files


Batches: 100%|██████████| 11/11 [00:08<00:00,  1.33it/s]


(44, 1024)
What time is breakfast?
    0.59 restaurant.txt
    0.58 room_service.txt
    0.58 breakfast.txt
How much is the spa?
    0.67 spa.txt
    0.54 hotel_facilities.txt
    0.52 pricing_and_rates.txt
Can I bring my dog?
    0.62 pet_policy.txt
    0.5 gym.txt
    0.5 accessibility.txt
الفطار لحد إمتى؟
    0.62 restaurant.txt
    0.58 breakfast.txt
    0.57 room_service.txt
ممكن أدخل الأوضة بدري؟
    0.37 identification_policy.txt
    0.35 pet_policy.txt
    0.35 payment_methods.txt
What's the weather tomorrow?
    0.41 noise_policy.txt
    0.4 special_requests.txt
    0.4 restaurant.txt
time: 0.024 seconds
